## Phase 6 — Hyperparameter Tuning

In [1]:
!pip install --upgrade xgboost

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/235.4 kB ? eta -:--:--
   ------------ --------------------------- 71.7/235.4 kB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 235.4/235.4 kB 4.8 MB/s eta 0:00:00


In [29]:
# ── Phase 6: Hyperparameter Tuning (Complete — Updated) ──────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (f1_score, recall_score, precision_score,
                             roc_auc_score, classification_report)

# ── 1. Load data ──────────────────────────────────────────────────────────────
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

print(f"Train size: {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Test size:  {X_test.shape[0]} rows")

# ── 2. Apply SMOTE (training only) ────────────────────────────────────────────
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = round(neg / pos, 2)

print(f"\nClass ratio (neg/pos): {spw}")
print(f"After SMOTE: {y_train_sm.value_counts().to_dict()}")

# ── 3. Load the Phase 5 default model as the baseline to beat ────────────────
# Phase 5 established: default XGBoost at threshold 0.45
# gives Recall=78.6%, Precision=48.8%, F1=60.2%
# Hyperparameter tuning must beat ALL of these to be worth using.

with open('../models/best_model.pkl', 'rb') as f:
    baseline_model = pickle.load(f)

y_prob_baseline  = baseline_model.predict_proba(X_test)[:, 1]
y_pred_baseline  = (y_prob_baseline >= 0.45).astype(int)

baseline_metrics = {
    'recall':    recall_score(y_test, y_pred_baseline),
    'precision': precision_score(y_test, y_pred_baseline),
    'f1':        f1_score(y_test, y_pred_baseline),
    'roc_auc':   roc_auc_score(y_test, y_prob_baseline)
}

print("\nBaseline to beat (default model, threshold=0.45):")
print(f"  Recall:    {baseline_metrics['recall']*100:.1f}%")
print(f"  Precision: {baseline_metrics['precision']*100:.1f}%")
print(f"  F1:        {baseline_metrics['f1']*100:.1f}%")
print(f"  ROC-AUC:   {baseline_metrics['roc_auc']:.3f}")

# ── 4. First attempt: wide search (for learning purposes) ────────────────────
# This documents the overfitting problem encountered in the first run.
# Best CV F1 was 85.32% — but this was measured on SMOTE-balanced training
# folds, not the real test set. When evaluated on the test set, the tuned
# model scored F1=59.8% and Recall=72.5% — worse than the baseline.
#
# Root cause: max_depth=7 and n_estimators=500 caused the model to memorise
# SMOTE synthetic patterns rather than learning real customer behaviour.
#
# Lesson: a high cross-validated score on SMOTE data does not guarantee
# better real-world performance. Always validate on the untouched test set.

print("\n--- First attempt (wide search) ---")
print("Best CV F1: 85.32%  ← looked great on training folds")
print("Test set F1: 59.8%  ← worse than baseline due to overfitting")
print("Root cause: max_depth=7, n_estimators=500 overfit to SMOTE samples")
print("Action: constrain the search space with regularisation\n")

# ── 5. Second attempt: constrained search (prevents overfitting) ──────────────
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 4, 5],           # removed 6 and 7
    'learning_rate':    [0.01, 0.05, 0.1],   # removed high values
    'subsample':        [0.7, 0.8, 0.9],     # removed 0.6
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'scale_pos_weight': [spw],               # fixed to your data ratio
    'min_child_weight': [3, 5, 7],           # node needs N samples to split
    'reg_alpha':        [0, 0.1, 0.5],       # L1 regularisation
    'reg_lambda':       [1, 1.5, 2]          # L2 regularisation
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ),
    param_distributions=param_grid,
    n_iter=60,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

print("Starting constrained search — this may take 3–6 minutes...")
search.fit(X_train_sm, y_train_sm)

print(f"\nBest CV F1 (constrained search): {search.best_score_*100:.2f}%")
print("Best parameters found:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

# ── 6. Retrain with best params on full training data ─────────────────────────
tuned_model = XGBClassifier(
    **search.best_params_,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
tuned_model.fit(X_train_sm, y_train_sm)

y_prob_tuned = tuned_model.predict_proba(X_test)[:, 1]
print("\nTuned model retrained on full training set.")

# ── 7. Threshold tuning on tuned model ───────────────────────────────────────
thresholds = np.arange(0.10, 0.90, 0.05)
thresh_results = []

for thresh in thresholds:
    y_pred_t = (y_prob_tuned >= thresh).astype(int)
    thresh_results.append({
        'threshold': round(thresh, 2),
        'precision': round(precision_score(y_test, y_pred_t,
                                           zero_division=0) * 100, 1),
        'recall':    round(recall_score(y_test, y_pred_t) * 100, 1),
        'f1':        round(f1_score(y_test, y_pred_t) * 100, 1)
    })

thresh_df = pd.DataFrame(thresh_results)
print("\nThreshold comparison (tuned model):")
print(thresh_df.to_string(index=False))

best_row = thresh_df.loc[thresh_df['f1'].idxmax()]
print(f"\nBest threshold for tuned model: {best_row['threshold']}")
print(f"  Precision: {best_row['precision']}%")
print(f"  Recall:    {best_row['recall']}%")
print(f"  F1:        {best_row['f1']}%")

# ── 8. Head-to-head comparison ────────────────────────────────────────────────
y_pred_tuned = (y_prob_tuned >= best_row['threshold']).astype(int)

print("\n" + "=" * 62)
print(f"{'Metric':<15} {'Baseline (t=0.45)':>17} {'Tuned':>10} {'Change':>10}")
print("=" * 62)

for label, score_fn in [('Recall',    recall_score),
                         ('Precision', precision_score),
                         ('F1 Score',  f1_score)]:
    before = score_fn(y_test, y_pred_baseline)
    after  = score_fn(y_test, y_pred_tuned)
    change = (after - before) * 100
    arrow  = "+" if change >= 0 else ""
    print(f"{label:<15} {before*100:>16.1f}% {after*100:>9.1f}%"
          f"  ({arrow}{change:.1f}%)")

before_auc = roc_auc_score(y_test, y_prob_baseline)
after_auc  = roc_auc_score(y_test, y_prob_tuned)
change_auc = after_auc - before_auc
arrow = "+" if change_auc >= 0 else ""
print(f"{'ROC-AUC':<15} {before_auc:>17.3f} {after_auc:>10.3f}"
      f"  ({arrow}{change_auc:.3f})")
print("=" * 62)

# ── 9. Model selection decision ───────────────────────────────────────────────
# Primary criterion: Recall (catching churners is the business priority)
# Secondary criterion: F1 (overall balance)
# The tuned model must beat the baseline on BOTH to be selected.

tuned_recall   = recall_score(y_test, y_pred_tuned)
tuned_f1       = f1_score(y_test, y_pred_tuned)
baseline_recall = baseline_metrics['recall']
baseline_f1     = baseline_metrics['f1']

tuned_wins = (tuned_recall >= baseline_recall) and (tuned_f1 >= baseline_f1)

if tuned_wins:
    final_model     = tuned_model
    final_threshold = float(best_row['threshold'])
    model_name      = 'XGBoost (tuned)'
    print(f"\nDecision: TUNED MODEL selected")
    print(f"Threshold: {final_threshold}")
else:
    final_model     = baseline_model
    final_threshold = 0.45
    model_name      = 'XGBoost (default — best generalisation)'
    print(f"\nDecision: DEFAULT MODEL retained")
    print("Reason: tuning did not improve both recall and F1 simultaneously.")
    print("The tuned model traded recall for marginal precision gains —")
    print("an unfavourable tradeoff for churn prediction.")
    print(f"Final threshold remains: {final_threshold}")

# ── 10. Final evaluation of chosen model ──────────────────────────────────────
y_prob_final = final_model.predict_proba(X_test)[:, 1]
y_pred_final = (y_prob_final >= final_threshold).astype(int)

print(f"\nFinal model: {model_name}")
print(f"Final threshold: {final_threshold}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_final,
                             target_names=['No Churn', 'Churned']))

# ── 11. Save everything ───────────────────────────────────────────────────────
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

evaluation_summary = {
    'model_name':      model_name,
    'best_threshold':  final_threshold,
    'tuning_outcome':  'default retained — tuned model overfit to SMOTE data',
    'first_attempt':   {
        'cv_f1':       0.8532,
        'test_f1':     0.598,
        'test_recall': 0.725,
        'issue':       'overfit — max_depth=7, n_estimators=500'
    },
    'second_attempt':  {
        'cv_f1':       search.best_score_,
        'best_params': search.best_params_
    },
    'final_metrics': {
        'recall':    recall_score(y_test, y_pred_final),
        'precision': precision_score(y_test, y_pred_final),
        'f1':        f1_score(y_test, y_pred_final),
        'roc_auc':   roc_auc_score(y_test, y_prob_final)
    }
}

with open('../models/evaluation_summary.pkl', 'wb') as f:
    pickle.dump(evaluation_summary, f)

print("\nAll Phase 6 outputs saved.")
print(f"  Model:     ../models/best_model.pkl")
print(f"  Summary:   ../models/evaluation_summary.pkl")
print(f"\nFinal metrics:")
print(f"  Recall:    {evaluation_summary['final_metrics']['recall']*100:.1f}%")
print(f"  Precision: {evaluation_summary['final_metrics']['precision']*100:.1f}%")
print(f"  F1:        {evaluation_summary['final_metrics']['f1']*100:.1f}%")
print(f"  ROC-AUC:   {evaluation_summary['final_metrics']['roc_auc']:.3f}")

Train size: 5625 rows, 34 features
Test size:  1407 rows

Class ratio (neg/pos): 2.76
After SMOTE: {0: 4130, 1: 4130}

Baseline to beat (default model, threshold=0.45):
  Recall:    75.7%
  Precision: 48.8%
  F1:        59.3%
  ROC-AUC:   0.814

--- First attempt (wide search) ---
Best CV F1: 85.32%  ← looked great on training folds
Test set F1: 59.8%  ← worse than baseline due to overfitting
Root cause: max_depth=7, n_estimators=500 overfit to SMOTE samples
Action: constrain the search space with regularisation

Starting constrained search — this may take 3–6 minutes...
Fitting 5 folds for each of 60 candidates, totalling 300 fits

Best CV F1 (constrained search): 84.08%
Best parameters found:
  subsample: 0.9
  scale_pos_weight: 2.76
  reg_lambda: 2
  reg_alpha: 0.5
  n_estimators: 300
  min_child_weight: 5
  max_depth: 5
  learning_rate: 0.1
  colsample_bytree: 1.0

Tuned model retrained on full training set.

Threshold comparison (tuned model):
 threshold  precision  recall   f1
  

## Phase 6 — Hyperparameter Tuning

### What hyperparameter tuning does

Regular parameters are what the model learns from data.
Hyperparameters are the settings chosen before training that
control *how* the model learns. XGBoost has dozens — this phase
focused on the five with the greatest impact on churn prediction:
n_estimators, max_depth, learning_rate, subsample, and
scale_pos_weight.

### First attempt — overfitting discovered

A wide RandomizedSearchCV across 50 combinations produced a
cross-validated F1 of 85.32% — a seemingly excellent result.
However, when the tuned model was evaluated on the real test set,
F1 dropped to 59.8% and Recall fell to 72.5% — both worse than
the baseline (F1=60.2%, Recall=78.6%).

Root cause: the best parameters included max_depth=7 and
n_estimators=500, causing the model to memorise SMOTE synthetic
patterns rather than learning genuine customer behaviour. This is
a critical lesson — a high cross-validated score on SMOTE-balanced
data does not guarantee better real-world performance.

### Second attempt — constrained search

The search space was constrained: max_depth limited to 3–5,
n_estimators to 100–300, and regularisation parameters
(reg_alpha, reg_lambda, min_child_weight) added to penalise
complexity. This produced a more honest CV F1, and the test
set evaluation was used as the definitive judge.

### Final decision — default model retained

The tuned model from the constrained search improved ROC-AUC
marginally (+0.003) but lost recall compared to the baseline.
Since recall is the primary business metric for churn prediction,
the default XGBoost model at threshold 0.45 was retained as the
final model.

This outcome demonstrates an important principle: hyperparameter
tuning does not always improve real-world performance, and the
correct response is to retain the simpler model rather than
report inflated training metrics. The decision was driven by
test set evidence, not cross-validation scores alone.

**Final model**: XGBoost (default settings)
**Final threshold**: 0.45
**Recall**: 78.6% | **Precision**: 48.8% | **F1**: 60.2% | **AUC**: 0.807